Imports and environment variables

In [42]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import math
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from arch import arch_model
from scipy.stats import norm

GARCH - META stock price extraction and plot

In [43]:
meta = yf.download("META", start="2020-01-01")
prices = meta['Close']['META']
log_returns = np.log(prices / prices.shift(1))
# AR(1)
ar1 = ARIMA(log_returns.dropna(), order=(1,0,0)).fit()
# MA(1)
ma1 = ARIMA(log_returns.dropna(), order=(0,0,1)).fit()
# ARMA(1,1)
arma11 = ARIMA(log_returns.dropna(), order=(1,0,1)).fit()
print('AIC of AR(1) : ' ,ar1.aic)
print('AIC of MA(1) : ',ma1.aic)
print('AIC of ARMA(1,1) : ',arma11.aic)
LR = 2 * (arma11.llf - ar1.llf)
from scipy.stats import chi2
p_value = 1 - chi2.cdf(LR, 1)
print(p_value)

[*********************100%***********************]  1 of 1 completed
c:\Users\myamo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\myamo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\myamo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\myamo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\statsmode

AIC of AR(1) :  -6951.297805584511
AIC of MA(1) :  -6951.208585188294
AIC of ARMA(1,1) :  -6950.098819231076
0.3707904785778452


c:\Users\myamo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\myamo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\myamo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


lowest AIC is achieved my AR(1) model, We get a large p-value of 1. Thus we fail to reject the null hypothesis which suggests that the ARMA(1,1) model is a justified complexification of the simpler model AR(1). 
AR(1) is a sufficient simplification of the more complex model.

In [44]:
# Normal
ar1_garch11_std = arch_model(log_returns.dropna(), mean='AR', lags = 1, vol='GARCH', p=1, q=1).fit()
# Student t
ar1_garch11_stt = arch_model(log_returns.dropna(), mean='AR', lags = 1, vol='GARCH', p=1, q=1, dist="t").fit()
# GED
ar1_garch11_ged = arch_model(log_returns.dropna(), mean='AR', lags = 1, vol='GARCH', p=1, q=1, dist="ged").fit()
# Skewed Student t
ar1_garch11_sstt = arch_model(log_returns.dropna(), mean='AR', lags = 1, vol='GARCH', p=1, q=1, dist="skewt").fit()
print('AIC of Gaussian) : ' ,ar1_garch11_std.aic)
print('AIC of Student S) : ',ar1_garch11_stt.aic)
print('AIC of GED : ',ar1_garch11_ged.aic)
print('AIC of Skewed Student S) : ',ar1_garch11_sstt.aic)


Iteration:      1,   Func. Count:      7,   Neg. LLF: 512073581.5041787
Iteration:      2,   Func. Count:     20,   Neg. LLF: 33647.23805595086
Iteration:      3,   Func. Count:     32,   Neg. LLF: 1190289.767835591
Iteration:      4,   Func. Count:     44,   Neg. LLF: 1592375.1580711494
Iteration:      5,   Func. Count:     57,   Neg. LLF: 97747.94157613533
Iteration:      6,   Func. Count:     69,   Neg. LLF: 11670.472628919091
Iteration:      7,   Func. Count:     80,   Neg. LLF: 26953137.195376217
Iteration:      8,   Func. Count:     93,   Neg. LLF: 9278324.898622401
Iteration:      9,   Func. Count:    107,   Neg. LLF: 16287.393611635853
Iteration:     10,   Func. Count:    118,   Neg. LLF: 422835.5610548721
Iteration:     11,   Func. Count:    129,   Neg. LLF: 962672.4879717138
Iteration:     12,   Func. Count:    141,   Neg. LLF: 4657215.110316838
Iteration:     13,   Func. Count:    154,   Neg. LLF: 25471.80520732673
Iteration:     14,   Func. Count:    165,   Neg. LLF: 7156.5

c:\Users\myamo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\arch\univariate\base.py:694: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.0007657. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  self._check_scale(resids)
c:\Users\myamo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\arch\univariate\base.py:694: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.0007657. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  self._check_scale(re

Lowest AIC achieved by the GEV Model : -7356.014133445697


In [45]:
print(ar1_garch11_ged.summary())

                                 AR - GARCH Model Results                                 
Dep. Variable:                               META   R-squared:                       0.001
Mean Model:                                    AR   Adj. R-squared:                  0.001
Vol Model:                                  GARCH   Log-Likelihood:                3687.08
Distribution:      Generalized Error Distribution   AIC:                          -7362.15
Method:                        Maximum Likelihood   BIC:                          -7329.87
                                                    No. Observations:                 1603
Date:                            Thu, May 21 2026   Df Residuals:                     1601
Time:                                    16:46:57   Df Model:                            2
                                  Mean Model                                 
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-------------------------